In [1]:
import json
from tqdm import tqdm
import numpy as np
# import time

from pyspark.sql.functions import when, col, current_timestamp
from pyspark.sql import SparkSession

from kafka import KafkaConsumer, KafkaProducer
from config import kafka_config

In [2]:
# Налаштування конфігурації SQL бази даних
jdbc_url = "jdbc:mysql://217.61.57.46:3306/olympic_dataset"
# jdbc_table = "athlete_bio"
jdbc_user = "neo_data_admin"
jdbc_password = "Proyahaxuqithab9oplp"

# Створення Spark сесії
spark = SparkSession.builder \
    .config("spark.jars", "mysql-connector-j-8.0.32.jar") \
    .appName("JDBCToKafka") \
    .getOrCreate()


In [3]:
# Етап 3.а): Зчитати дані з mysql таблиці athlete_event_results

# Читання result-даних з SQL бази даних
res_df = spark.read.format('jdbc').options(
    url=jdbc_url,
    driver='com.mysql.cj.jdbc.Driver',  # com.mysql.jdbc.Driver
    dbtable="athlete_event_results",
    user=jdbc_user,
    password=jdbc_password) \
    .load()

In [4]:
# Перетворіть DataFrame у JSON
json_result_data = res_df.toJSON().collect()
print(f'{len(json_result_data)} rows for sending')

316834 rows for sending


In [5]:
# Етап 3.б): записати в кафка топік athlete_event_results

# Створення Kafka Producer
producer = KafkaProducer(
    bootstrap_servers=kafka_config['bootstrap_servers'],
    security_protocol=kafka_config['security_protocol'],
    sasl_mechanism=kafka_config['sasl_mechanism'],
    sasl_plain_username=kafka_config['username'],
    sasl_plain_password=kafka_config['password'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# контрольний ідентифікатор сесії передачі датасета через топік
CONTROL_SESSION_ID = 2

# Назва топіку
RESULT_TOPIC_NAME = 'athlete_event_results'

# Загальна кількість рядків для передачі
session_lenght = len(json_result_data)

# Службова інформація для контролю номеру сессії та повноти отримання переданих даних
headers = [
            ('session_id', str(CONTROL_SESSION_ID).encode('utf-8')),
            ('session_total_row_number', str(session_lenght).encode('utf-8'))
        ]

# Кількість рядків, одночасно передаються
step = 4000

# Відправлення повідомлення в топік
try:
    for i in tqdm(range(int(np.ceil(session_lenght/step)))):
        start_position = i * step
        finish_pisition = i * step + step
        
        my_header = headers.copy()
        my_header.append(('start_position', str(start_position).encode('utf-8')))
        # my_header.append(('finish_position', str(finish_pisition).encode('utf-8')))
        
        # print(json_result_data[i])
        data = list(map(json.loads, json_result_data[start_position:finish_pisition]))
        # data = json_result_data[i]
        # print(data)
        producer.send(RESULT_TOPIC_NAME, 
                      # key=str(i), 
                      value=data,
                      headers=my_header
                     )
        producer.flush()  # Очікування, поки всі повідомлення будуть відправлені
        # print(f"Message #{i} sent to topic '{result_topic_name}' successfully.")
        # # time.sleep(2)
except Exception as e:
    print(f"An error occurred: {e}")
finally:   
    producer.close()  # Закриття producer

100%|██████████| 80/80 [01:02<00:00,  1.29it/s]
